In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    classification_report,
    confusion_matrix
)
from IPython.display import display

# Project paths
project_root = Path(r"D:\Smart bin classifier")
sys.path.insert(0, str(project_root))

test_image_dir = project_root / "data" / "processed" / "test"
test_labels_path = project_root / "data" / "processed" / "test_labels.csv"
catalogue_path = project_root / "catalogue.csv"
model_dir = project_root / "models"

# Check that the required files exist
required_paths = [
    test_image_dir,
    test_labels_path,
    catalogue_path,
    model_dir / "resnet18_finetuned_best.pth",
    model_dir / "positive_quantity_models.joblib"
]

for path in required_paths:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

# Load labels and catalogue
test_labels = pd.read_csv(test_labels_path)
catalogue = pd.read_csv(catalogue_path)
products = catalogue["asin"].astype(str).tolist()

print("\nDevice:", "CUDA GPU" if torch.cuda.is_available() else "CPU")
print("Test images:", len(list(test_image_dir.glob("*.jpg"))))
print("Test label rows:", len(test_labels))
print("Catalogue products:", len(products))

display(catalogue[["asin", "product_name"]])

FOUND: D:\Smart bin classifier\data\processed\test
FOUND: D:\Smart bin classifier\data\processed\test_labels.csv
FOUND: D:\Smart bin classifier\catalogue.csv
FOUND: D:\Smart bin classifier\models\resnet18_finetuned_best.pth
FOUND: D:\Smart bin classifier\models\positive_quantity_models.joblib

Device: CUDA GPU
Test images: 200
Test label rows: 200
Catalogue products: 5


,asin,product_name
0,B00O4OR4GQ,Fujifilm Instax Mini Instant Film (3 Twin Pack...
1,B013UJS35G,TaoTronics TT-AH002 30W Ultrasonic Humidifier ...
2,B0159A0LH2,Pumpkin Pie Quest Bars 2 Pack Total of 24 Coat...
3,B019D9HESO,RTIC 30 oz. Tumbler
4,B01B1GE6KG,Laser Lights Outdoor Starry Projector Red and ...


In [4]:
from src.predict import load_models, predict_image
from PIL import Image

# Load the trained models; do not retrain
presence_model, feature_extractor, quantity_models = load_models()

# Index the test labels by filename for later evaluation
test_labels = test_labels.set_index("filename")

prediction_rows = []

# Predict each test image using image pixels only
for i, image_path in enumerate(sorted(test_image_dir.glob("*.jpg")), start=1):
    if i % 25 == 0:
        print(f"Processed {i}/200 images")
    result = predict_image(
        image_path,
        presence_model,
        feature_extractor,
        quantity_models
    )

    row = {"filename": image_path.name}

    for asin in products:
        row[f"qty_{asin}"] = result["quantities"][asin]

    prediction_rows.append(row)

test_predictions = pd.DataFrame(prediction_rows).set_index("filename")

# Align actual labels and predictions by filename
actual_quantities = test_labels.loc[
    test_predictions.index,
    [f"qty_{asin}" for asin in products]
].astype(int)

predicted_quantities = test_predictions[
    [f"qty_{asin}" for asin in products]
].astype(int)

print("Test images predicted:", len(test_predictions))
print("Prediction shape:", predicted_quantities.shape)
print("Actual label shape:", actual_quantities.shape)
print("\nSample predictions:")
display(test_predictions.head())

Processed 25/200 images
Processed 50/200 images
Processed 75/200 images
Processed 100/200 images
Processed 125/200 images
Processed 150/200 images
Processed 175/200 images
Processed 200/200 images
Test images predicted: 200
Prediction shape: (200, 5)
Actual label shape: (200, 5)

Sample predictions:


,qty_B00O4OR4GQ,qty_B013UJS35G,qty_B0159A0LH2,qty_B019D9HESO,qty_B01B1GE6KG
filename,,,,,
07296.jpg,0,0,1,0,0
07320.jpg,0,1,0,0,0
09001.jpg,0,0,0,0,0
102685.jpg,0,0,0,0,0
104894.jpg,0,0,1,0,0


In [5]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    precision_recall_fscore_support
)

# 1. Exact-match accuracy: all five quantities must match
exact_match = (
    actual_quantities.to_numpy() == predicted_quantities.to_numpy()
).all(axis=1)

exact_match_accuracy = exact_match.mean()

# 2. Mean Absolute Error across all 200 x 5 quantity predictions
quantity_mae = mean_absolute_error(
    actual_quantities.to_numpy().ravel(),
    predicted_quantities.to_numpy().ravel()
)

print("FINAL TEST RESULTS")
print("=" * 45)
print(f"Test images: {len(actual_quantities)}")
print(f"Exact-match accuracy: {exact_match_accuracy:.4f}")
print(f"Quantity MAE: {quantity_mae:.4f}")

# 3. Product-presence metrics (quantity > 0 means present)
actual_presence = (actual_quantities.to_numpy() > 0).astype(int)
predicted_presence = (predicted_quantities.to_numpy() > 0).astype(int)

precision, recall, f1, support = precision_recall_fscore_support(
    actual_presence,
    predicted_presence,
    average="macro",
    zero_division=0
)

print(f"\nMacro presence precision: {precision:.4f}")
print(f"Macro presence recall:    {recall:.4f}")
print(f"Macro presence F1-score:  {f1:.4f}")

# 4. Per-product results
product_metrics = []

for asin in products:
    actual = actual_quantities[f"qty_{asin}"].to_numpy()
    predicted = predicted_quantities[f"qty_{asin}"].to_numpy()

    product_metrics.append({
        "ASIN": asin,
        "Exact-match accuracy": (actual == predicted).mean(),
        "MAE": mean_absolute_error(actual, predicted),
        "Actual positive bins": int((actual > 0).sum()),
        "Predicted positive bins": int((predicted > 0).sum())
    })

product_metrics_df = pd.DataFrame(product_metrics)
display(product_metrics_df.round(4))

FINAL TEST RESULTS
Test images: 200
Exact-match accuracy: 0.6300
Quantity MAE: 0.0950

Macro presence precision: 0.8487
Macro presence recall:    0.7545
Macro presence F1-score:  0.7909


,ASIN,Exact-match accuracy,MAE,Actual positive bins,Predicted positive bins
0,B00O4OR4GQ,0.865,0.200,38,30
1,B013UJS35G,0.955,0.045,35,40
2,B0159A0LH2,0.965,0.045,30,28
3,B019D9HESO,0.905,0.100,19,19
4,B01B1GE6KG,0.930,0.085,31,20


In [6]:
# Find test bins where at least one product quantity was predicted incorrectly
wrong_mask = (
    actual_quantities.to_numpy() != predicted_quantities.to_numpy()
).any(axis=1)

wrong_filenames = actual_quantities.index[wrong_mask]

print("Incorrectly predicted test bins:", len(wrong_filenames))

# Select five examples with the most quantity mismatches
mismatch_counts = (
    actual_quantities.loc[wrong_filenames]
    .ne(predicted_quantities.loc[wrong_filenames])
    .sum(axis=1)
)

five_failures = mismatch_counts.sort_values(
    ascending=False
).head(5).index

# Show actual and predicted quantities side by side
failure_rows = []

for filename in five_failures:
    row = {"filename": filename}

    for asin in products:
        row[f"actual_{asin}"] = int(
            actual_quantities.loc[filename, f"qty_{asin}"]
        )
        row[f"predicted_{asin}"] = int(
            predicted_quantities.loc[filename, f"qty_{asin}"]
        )

    row["number_of_quantity_errors"] = int(
        mismatch_counts.loc[filename]
    )
    failure_rows.append(row)

failure_df = pd.DataFrame(failure_rows)

display(failure_df)

Incorrectly predicted test bins: 74


,filename,actual_B00O4OR4GQ,predicted_B00O4OR4GQ,actual_B013UJS35G,predicted_B013UJS35G,actual_B0159A0LH2,predicted_B0159A0LH2,actual_B019D9HESO,predicted_B019D9HESO,actual_B01B1GE6KG,predicted_B01B1GE6KG,number_of_quantity_errors
0,437896.jpg,1,0,0,0,0,0,0,1,0,0,2
1,275440.jpg,0,0,0,0,1,0,0,1,0,0,2
2,10593.jpg,0,0,0,1,0,0,0,0,0,0,1
3,108102.jpg,0,0,0,0,0,0,1,0,0,0,1
4,111960.jpg,0,0,0,0,2,0,0,0,0,0,1


In [7]:
# Display only the products with incorrect quantities in each selected bin
for filename in five_failures:
    print(f"\nBin image: {filename}")

    for asin in products:
        actual = int(actual_quantities.loc[filename, f"qty_{asin}"])
        predicted = int(predicted_quantities.loc[filename, f"qty_{asin}"])

        if actual != predicted:
            product_name = catalogue.loc[
                catalogue["asin"] == asin, "product_name"
            ].iloc[0]

            print(
                f"  {product_name}\n"
                f"    Actual quantity: {actual}\n"
                f"    Predicted quantity: {predicted}"
            )


Bin image: 437896.jpg
  Fujifilm Instax Mini Instant Film (3 Twin Packs, 60 Total Pictures) Value Set
    Actual quantity: 1
    Predicted quantity: 0
  RTIC 30 oz. Tumbler
    Actual quantity: 0
    Predicted quantity: 1

Bin image: 275440.jpg
  Pumpkin Pie Quest Bars 2 Pack Total of 24 Coated Protein Bars Delicious Limited Time
    Actual quantity: 1
    Predicted quantity: 0
  RTIC 30 oz. Tumbler
    Actual quantity: 0
    Predicted quantity: 1

Bin image: 10593.jpg
  TaoTronics TT-AH002 30W Ultrasonic Humidifier with Cool Mist, Classic Dial Knob Control, 3.5L Large Capacity, Two 360 degree Rotatable Outlets
    Actual quantity: 0
    Predicted quantity: 1

Bin image: 108102.jpg
  RTIC 30 oz. Tumbler
    Actual quantity: 1
    Predicted quantity: 0

Bin image: 111960.jpg
  Pumpkin Pie Quest Bars 2 Pack Total of 24 Coated Protein Bars Delicious Limited Time
    Actual quantity: 2
    Predicted quantity: 0


## 5. Failure Analysis

### 5.1 Summary

The model correctly predicted all five catalogue-product quantities for 126 of the 200 test bins, achieving an exact-match accuracy of **63%**. The remaining 74 bins had at least one incorrect quantity prediction.

### 5.2 Selected Incorrect Predictions

| Image ID | Product | Actual Quantity | Predicted Quantity | Error |
|---|---|---:|---:|---|
| 437896.jpg | Fujifilm Instax Mini Film | 1 | 0 | False negative |
| 437896.jpg | RTIC 30 oz. Tumbler | 0 | 1 | False positive |
| 275440.jpg | Pumpkin Pie Quest Bars | 1 | 0 | False negative |
| 275440.jpg | RTIC 30 oz. Tumbler | 0 | 1 | False positive |
| 10593.jpg | TaoTronics Humidifier | 0 | 1 | False positive |
| 108102.jpg | RTIC 30 oz. Tumbler | 1 | 0 | False negative |
| 111960.jpg | Pumpkin Pie Quest Bars | 2 | 0 | False negative |

These examples include both missed products and products incorrectly predicted as present. The last example also demonstrates a quantity estimation failure: the model predicted zero when the actual quantity was two.

### 5.3 Possible Causes

The numerical predictions alone do not establish the visual cause of each error. Potential causes to investigate by inspecting the corresponding images include:

- **Visual similarity:** Different products may share shapes, colours, or packaging characteristics.
- **Occlusion:** Products may be partly hidden by other objects, packaging, or tape.
- **Product appearance:** Lighting, camera angle, and image quality may affect visual features.
- **Quantity estimation:** Multiple units may be difficult to distinguish, particularly when packaged together.
- **Limited examples:** Some products and non-zero quantities occur less frequently than others in the training data.

These are hypotheses to investigate, not confirmed explanations for the listed images.

### 5.4 Improvements

1. Inspect the five images alongside their ground-truth labels to determine the actual visual causes.
2. Evaluate false negatives and false positives separately for each catalogue product.
3. Review class imbalance and consider suitable training-time weighting or sampling.
4. Add more representative training examples for difficult appearances and quantities, where permitted.
5. Compare any proposed changes using the validation set. Keep the test set reserved for final evaluation.

### 5.5 Conclusion

The model performs better on product presence than a simple guess would suggest, but its 63% exact-match accuracy shows that predicting every product quantity correctly in a bin remains challenging. The failure analysis identifies concrete cases for further investigation without using the test set to tune the model.